In [ ]:
import torch
from torch.nn.functional import pad, avg_pool2d
from einops import rearrange


In [ ]:
def pad_to_max_size(batch, patch_size):
    # Extract images and labels from the batch
    images, labels = zip(*batch)
    
    # Find the maximum height and width in the batch
    max_height = max(img.shape[1] for img in images)
    max_width = max(img.shape[2] for img in images)
    
    padded_images = []
    patch_masks = []
    for img in images:
        _, h, w = img.shape
        # Calculate padding
        padding = (0, max_width - w, 0, max_height - h)  # (left, right, top, bottom)
        padded_image = pad(img, padding, value=0)  # Pad with zeros (can change value if needed)
        padded_images.append(padded_image)

        # Create pixel-level mask
        pixel_mask = torch.zeros((max_height, max_width), dtype=torch.float32)
        pixel_mask[:h, :w] = 1.0  # Mark valid regions as 1
        
        # Downsample pixel-level mask to patch size
        pixel_mask = pixel_mask.unsqueeze(0)  # Add channel dimension for pooling
        patch_mask = avg_pool2d(pixel_mask, kernel_size=patch_size, stride=patch_size)
        patch_mask = (patch_mask > 0).int()  # Convert pooled mask to binary
        patch_masks.append(patch_mask)

    # Stack padded images and patch masks into tensors
    padded_images = torch.stack(padded_images)
    patch_masks = torch.stack(patch_masks).squeeze(1)  # Remove channel dimension from masks
    patch_masks = rearrange(patch_masks, "b h w -> b (h w)").float()

    # Convert labels to a tensor
    labels = torch.tensor(labels)
    
    return padded_images, patch_masks, labels

In [ ]:
# Add this as a new cell
import torch
import matplotlib.pyplot as plt

# Create sample data with different sized images
def create_sample_batch():
    # Create 3 images of different sizes
    img1 = torch.ones(3, 224, 224)  # Standard sized image
    img2 = torch.ones(3, 160, 320)  # Wide image
    img3 = torch.ones(3, 280, 180)  # Tall image
    
    # Add some pattern to easily distinguish the images
    img1[:, ::2, ::2] = 0  # Checkerboard pattern
    img2[:, :, ::3] = 0    # Vertical stripes
    img3[:, ::3, :] = 0    # Horizontal stripes
    
    # Create corresponding labels
    labels = [0, 1, 2]
    
    # Create batch
    batch = list(zip([img1, img2, img3], labels))
    return batch

# Test the padding function
batch = create_sample_batch()
patch_size = 2
padded_images, patch_masks, labels = pad_to_max_size(batch, patch_size)

# Visualize results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Original vs Padded Images with Patch Masks')

for i in range(3):
    # Show original image
    axes[0, i].imshow(batch[i][0].permute(1, 2, 0))
    axes[0, i].set_title(f'Original Image {i+1}\nShape: {batch[i][0].shape}')
    axes[0, i].axis('off')
    
    # Show patch mask
    h, w = padded_images[i].shape[1:]
    patch_mask_reshaped = patch_masks[i].view(h//patch_size, w//patch_size)
    axes[1, i].imshow(patch_mask_reshaped, cmap='gray')
    axes[1, i].set_title(f'Patch Mask {i+1}\nShape: {patch_mask_reshaped.shape}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("Padded images shape:", padded_images.shape)
print("Patch masks shape:", patch_masks.shape)
print("Labels:", labels)